# Render en la T4 de ColabEsta notebook corre **los mismos dos renders** que ya estan medidos en CPU en elrepo, pero en la placa de Colab, para saber cuanto se gana de verdad. Losnumeros a batir, medidos en el contenedor de la sesion (4 nucleos, sin placa):| escena | en CPU ||---|---|| `casita.py` — 332 caras, Cycles 160 muestras, 1100x750 | **1 min 39 s** || `pelota.py` — un cuadro, 64 muestras, 800x450 | **6,5 s** || `pelota.py` — los 60 cuadros | **6 min 30 s** |Tres cosas antes de arrancar:1. **Poné la placa**: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.   Sin eso la celda 1 te lo va a decir y el resto renderiza en CPU.2. Colab gratis **se desconecta** por inactividad y tiene cupo diario de GPU. Lo   que quede en `/content` se pierde: lo que sirva, bajalo o mandalo al Drive.3. La sección 6 es **opcional**.Corré las celdas en orden. La 2 tarda 1-2 minutos (baja Blender); el resto es rápido.

## 1. ¿Qué placa tocó?

In [ ]:
import subprocesss = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",                    "--format=csv,noheader"], capture_output=True, text=True)if s.returncode == 0 and s.stdout.strip():    print("placa:", s.stdout.strip())else:    print("NO HAY PLACA. Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).")    print("Podés seguir igual, pero va a renderizar en CPU y no vas a comparar nada.")

## 2. Blender 4.3.2 y el repo**4.3.2 exactamente**, que es la version con la que estan medidos los tiempos dearriba. Con otra version los numeros dejan de ser comparables.El Blender de Debian que se usa en el contenedor viene **sin OpenImageDenoise** y**sin numpy**; este, el oficial de blender.org, trae los dos. Por eso acá eldenoise se podría prender — pero lo dejamos apagado igual, para estar comparandoel mismo render y no otro.

In [ ]:
%%bashset -ecd /content# Blender necesita estas para arrancar, aunque sea en --backgroundapt-get install -y -qq libxi6 libxxf86vm1 libxfixes3 libxrender1 libsm6 libgl1 2>/dev/null | tail -1if [ ! -x blender/blender ]; then  echo "bajando Blender 4.3.2..."  wget -q https://download.blender.org/release/Blender4.3/blender-4.3.2-linux-x64.tar.xz  tar xf blender-4.3.2-linux-x64.tar.xz  mv blender-4.3.2-linux-x64 blender  rm blender-4.3.2-linux-x64.tar.xzfi./blender/blender --version | head -1[ -d repo ] || git clone -q --depth 1 -b claude/new-session-8309f5 https://github.com/Juniorspro/New-General-Games-Assets repols repo/herramientas/blender/*.py

## 3. La casita — contra 1 min 39 s`DISPOSITIVO=GPU` es lo que hace que Cycles use la placa. Ojo con la trampa queresuelve el script: poner `cycles.device = "GPU"` **no alcanza** — si en laspreferencias no hay ningún aparato prendido, Cycles se cae a CPU y renderizaigual, sin avisar nada. Por eso imprime `PLACA:` con lo que realmente eligió. Siahí dice CPU, el número de abajo no es de la placa.

In [ ]:
import os, subprocess, timeBASE = 99          # 1 min 39 s medidos en CPU, 4 nucleos (herramientas/blender/LEEME.md)os.makedirs("/content/salida", exist_ok=True)entorno = dict(os.environ, DISPOSITIVO="GPU", SALIDA="/content/salida", MUESTRAS="160")t0 = time.time()p = subprocess.run(["/content/blender/blender", "--background", "--python",                    "/content/repo/herramientas/blender/casita.py"],                   env=entorno, capture_output=True, text=True)seg = time.time() - t0for l in p.stdout.splitlines():    if l.startswith(("PLACA:", "CASITA:", "Saved:")):        print(l)if p.returncode:    print("\n--- se rompio ---\n", p.stdout[-1500:], p.stderr[-1500:])else:    print(f"\ncasita: {seg:.1f} s en la placa  ·  {BASE} s en CPU  ·  x{BASE/seg:.1f}")

In [ ]:
from IPython.display import Image, displaydisplay(Image("/content/salida/casita.png", width=760))

## 4. La pelota — primero un cuadro, contra 6,5 sUn cuadro solo antes de largar los sesenta. Es la regla que ya está anotada en elrepo: elegir mal las muestras cuesta una hora de render.

In [ ]:
import os, subprocess, timeBASE = 6.5         # segundos por cuadro en CPU, 64 muestras, 800x450entorno = dict(os.environ, DISPOSITIVO="GPU", MOTOR="CYCLES", MUESTRAS="64",               ANCHO="800", ALTO="450", DESDE="1", HASTA="1",               RENDERIZAR="1", SALIDA="/tmp/pelota/f_")t0 = time.time()p = subprocess.run(["/content/blender/blender", "--background", "--python",                    "/content/repo/herramientas/blender/pelota.py"],                   env=entorno, capture_output=True, text=True)seg = time.time() - t0for l in p.stdout.splitlines():    if l.startswith(("PLACA:", "ESCENA LISTA", "Saved:")):        print(l)if p.returncode:    print("\n--- se rompio ---\n", p.stdout[-1500:], p.stderr[-1500:])else:    print(f"\nun cuadro: {seg:.1f} s en la placa  ·  {BASE} s en CPU  ·  x{BASE/seg:.1f}")    print(f"los 60 saldrian en ~{seg*60/60:.1f} min  ·  en CPU eran 6,5 min")

## 5. Los 60 cuadros y el video`armar_video.py` no vuelve a renderizar: lee los PNG con el secuenciador deBlender y los codifica en segundos. Volver a renderizar con salida FFMPEGcostaría los seis minutos y medio otra vez.

In [ ]:
import os, subprocess, time, globentorno = dict(os.environ, DISPOSITIVO="GPU", MOTOR="CYCLES", MUESTRAS="64",               ANCHO="800", ALTO="450", DESDE="1", HASTA="60",               RENDERIZAR="1", SALIDA="/tmp/pelota/f_")t0 = time.time()p = subprocess.run(["/content/blender/blender", "--background", "--python",                    "/content/repo/herramientas/blender/pelota.py"],                   env=entorno, capture_output=True, text=True)seg = time.time() - t0print(next((l for l in p.stdout.splitlines() if l.startswith("PLACA:")), "PLACA: ?"))print(f"60 cuadros: {seg/60:.1f} min en la placa  ·  6,5 min en CPU  ·  x{390/seg:.1f}")print("PNG escritos:", len(glob.glob('/tmp/pelota/f_*.png')))# el script del repo busca los PNG en /tmp/pelota/f_*.png y escribe /tmp/pelota_videov = subprocess.run(["/content/blender/blender", "--background", "--python",                    "/content/repo/herramientas/blender/armar_video.py"],                   capture_output=True, text=True)print(next((l for l in v.stdout.splitlines() if l.startswith("VIDEO LISTO")), v.stderr[-500:]))print(glob.glob("/tmp/pelota_video*"))

In [ ]:
import glob, base64from IPython.display import HTMLarch = glob.glob("/tmp/pelota_video*")[0]b64 = base64.b64encode(open(arch,"rb").read()).decode()HTML(f'<video width=640 controls src="data:video/mp4;base64,{b64}">')

---## 6. (Opcional) Claude Code adentro de este ColabEsto instala Claude Code **acá**, en la máquina que tiene la placa. Es otroagente: le hablás vos desde estas celdas.**La clave nunca va pegada en una celda** — la notebook se guarda en tu Drive contodo lo que escribas adentro. Va en *Secretos*: el ícono de la llave 🔑 en labarra izquierda → *Agregar secreto nuevo* → nombre `ANTHROPIC_API_KEY` → pegás elvalor → y prendés el interruptor *Acceso al notebook*.(Al margen: **no existe** ningún paquete `claude-colab` en PyPI. Claude Code seinstala por npm, que es lo que hace la celda de abajo.)

In [ ]:
%%bashnode --version 2>/dev/null || {  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1  apt-get install -y -qq nodejs}npm install -g @anthropic-ai/claude-code 2>&1 | tail -2claude --version

In [ ]:
import osfrom google.colab import userdataos.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")print("clave cargada:", len(os.environ["ANTHROPIC_API_KEY"]), "caracteres")

In [ ]:
# Una prueba corta, sin herramientas: solo que conteste.!claude -p "Contestá en una sola línea: ¿estás andando?"# Para que además pueda correr cosas, pedile los permisos que necesita y nada# más. Así ves qué toca:#   !claude -p "Decime qué placa hay" --allowed-tools "Bash(nvidia-smi:*)"